In [ ]:
!pip install -U langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4


In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END

In [ ]:
def mock_llm(state: MessagesState):
    return {"messages": [{"role":"ai", "content":"Hello world"}]}

graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)
graph = graph.compile()


In [ ]:
graph.invoke({"messages":[{"role":"user", "content": "hi!"}]})

{'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='50ceeb88-a732-423b-814d-446eafd7dc5f'),
  AIMessage(content='Hello world', additional_kwargs={}, response_metadata={}, id='8c84d78e-3390-4593-bc12-075aa50bdc95', tool_calls=[], invalid_tool_calls=[])]}

In [ ]:
!pip install langgraph langchain-groq langchain-core --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.6 MB/s eta 0:00:00


In [ ]:
import os, json, math
from typing import Annotated
from typing_extensions import TypedDict


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
@tool
def calculator(expression:str) -> str:
    "Evaluate mathematical expression usnig python math syntax"
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f'Error{e}'



In [ ]:
@tool
def web_search(query: str)-> str:
    "Search the web for factual information"
    db = {
        "population of france": "France has a population of approximately 68 million (2024).",
        "capital of australia": "The capital of Australia is Canberra, not Sydney.",
        "speed of light": "The speed of light in a vacuum is 299,792,458 metres per second.",
        "population of india": "India has a population of approximately 1.44 billion (2024).",
    }

    query_lower = query.lower().strip()
    for key, value in db.items():
        if key in query_lower or query_lower in key:
            return value
    return f'No result found for {query}'

In [ ]:
TOOLS = [calculator, web_search]

In [ ]:
llm = ChatGroq(
    api_key="....",
    model = "llama-3.3-70b-versatile",
    temperature = 0
)

In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)

In [ ]:
for t in TOOLS:
    print(t.name)

calculator
web_search


In [ ]:
llm_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x795b32085b20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x795b314a43b0>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'calculator', 'description': 'Evaluate mathematical expression usnig python math syntax', 'parameters': {'properties': {'expression': {'type': 'string'}}, 'required': ['expression'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'web_search', 'description': 'Search the web for factual information', 'parameters': {'properties': {'query': {'type': 'strin

## Multiple Schemas example


In [ ]:
class InputState(TypedDict):
    user_input: str

class OutputState(TypedDict):
    graph_output: str

class OverallState(TypedDict):
    foo: str
    user_input: str
    graph_output: str

class PrivateState(TypedDict):
    bar: str

def node_1(state: InputState)-> OverallState:
    return {"foo": state["user_input"] + "name"}

def node_2(state: OverallState) -> PrivateState
    return {"bar": state["foo"] + " is"}




## Build with graph

In [ ]:
# building Node 1: Agent Node
def agent_node(state: AgentState) -> dict:
    res = llm_with_tools.invoke(state['messages'])
    return {"messages": [res]}

In [ ]:
# tool node
tool_node = ToolNode(TOOLS)

In [ ]:
# ── Conditional edge: should we call tools or stop? ────
def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"   # route to tool_node
    return END           # no tool calls = final answer, stop


In [ ]:
graph_builder = StateGraph(AgentState)

In [ ]:

graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tool_node)


In [ ]:
# Entry point
graph_builder.add_edge(START, "agent")


In [ ]:
# After agent: check if we need tools or can stop
graph_builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END}
)

In [ ]:
# After tools: always go back to agent
graph_builder.add_edge("tools", "agent")


In [ ]:
# Compile — with MemorySaver for free persistence
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)


In [ ]:
print("Graph compiled successfully.")
print("Nodes:", list(graph_builder.nodes.keys()))

Graph compiled successfully.
Nodes: ['agent', 'tools']


In [ ]:
# run the graph

In [ ]:
def chat(question: str, thread_id: str ="default") -> str:
    config = {"configurable": {"thread_id" : thread_id}}

    result = graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config,

    )
    final = result["messages"][-1]
    return final.content

In [ ]:
answer = chat("What is the capital of Australia?", thread_id="test-1")
print(f"Answer: {answer}")


KeyError: 'message'

In [ ]:
!pip install langchain langchain_community langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    api_key="...",
    model = "llama-3.3-70b-versatile",
)

In [ ]:
res = llm.invoke("What is the capital of Australia?")
print(res)

content='The capital of Australia is Canberra.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.01294762, 'completion_tokens_details': None, 'prompt_time': 0.001471553, 'prompt_tokens_details': None, 'queue_time': 0.019209203, 'total_time': 0.014419173}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d675f-1ab8-7c51-9712-9a5800467b2c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50}


In [ ]:
r = llm.invoke("What is newly released movie in april 2026?")

In [ ]:
r

AIMessage(content="I'm not aware of any newly released movies in April 2026, as my knowledge cutoff is December 2023. I don't have real-time information or updates on future releases. However, I can suggest checking online movie databases, entertainment websites, or social media platforms for the latest information on upcoming movie releases in April 2026.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 46, 'total_tokens': 116, 'completion_time': 0.199318063, 'completion_tokens_details': None, 'prompt_time': 0.004480603, 'prompt_tokens_details': None, 'queue_time': 0.314717659, 'total_time': 0.203798666}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d675f-f1b7-74c0-a6ae-3ac1b3290808-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 70, 'total_

In [ ]:
from langchain.agents import create_agent

In [ ]:
from langchain_community.tools import TavilySearchResults

In [ ]:
search_tool = TavilySearchResults(search_depth="basic")

/tmp/ipykernel_20295/3131362913.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(search_depth="basic")


ValidationError: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [ ]:
agent = create_agent(
    tools = [search_tool],
    model= llm,
    # agent_type = "zero-shot-react-description",
    # verbose = True
)

NameError: name 'search_tool' is not defined

In [ ]:
agent.invoke("")

## Reflexion Agent System

In [1]:
!pip install langchain langchain_community langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [10]:
import datetime

In [28]:
# actor agent prompt
actor_agent_prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are exper AI Researcher.
            Current time: {time}

            1. {first_instruction}
            2. Reflect and critique your answer. Be severe to maximize improvement.
            3. After reflection, **list 1-3 search queries** separately for researching improvements. Do not include them inside the reflection.
            """,
        ),
        MessagesPlaceholder(variable_name="messages"),
        (
            "system",
            "Answer the user's question above using required format"
        )
    ]
).partial(
    time = lambda: datetime.datetime.now().isoformat(),
)

In [ ]:
## grounding the llm to provide the response in perticular json

In [6]:
from pydantic import BaseModel, Field
from typing import List

In [8]:
class Reflection(BaseModel):
    missing: str = Field(description="Critique of what is missing")
    superfluous: str = Field(description="Critique of what is superfluous")


In [9]:
class AnswerQuestion(BaseModel):
    answer: str = Field(
        description="approx 250 words details answer to question"
    )
    search_queries: List[str] = Field(
        description="1-3 search queris for researching improvements to address the critique of current answer"
    )
    reflection: Reflection = Field(
        description = "Your reflection on the initial answer"
    )

In [ ]:
# chain for responder

In [29]:
first_responder_prompt_template = actor_agent_prompt_template.partial(
    first_instruction = "Provide me a detailed 200 word answer"
)

In [14]:
from langchain_groq import ChatGroq

In [15]:
llm = ChatGroq(
    api_key="..",
    model = "llama-3.3-70b-versatile",
    temperature = 0
)

In [16]:
from langchain_core.output_parsers import PydanticOutputParser

In [20]:
from langchain_core.messages import HumanMessage

In [17]:
pydantic_parser = PydanticOutputParser(pydantic_object=AnswerQuestion)

In [32]:
first_responder_chain = first_responder_prompt_template | llm.bind_tools(
    tools = [AnswerQuestion], tool_choice="AnswerQuestion")

In [33]:
res=first_responder_chain.invoke({
    "messages":[HumanMessage(content="write me a blog post on how student can study effectively")]
})

In [36]:
import json

In [39]:
res

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '17ab1cha8', 'function': {'arguments': '{"answer":"To study effectively, students should start by setting clear goals and priorities. This involves identifying the most important topics to focus on, breaking down large tasks into smaller manageable chunks, and creating a schedule to stay organized. Active learning techniques such as summarizing notes in their own words, self-quizzing, and elaboration can also help deepen understanding and retention of material. Additionally, using visual aids like diagrams, flowcharts, and mind maps can make complex information more engaging and easier to review. Regular review and practice are crucial, with students aiming to review material within a day or two of initially learning it, and then at increasingly longer intervals to help solidify it in their long-term memory. Furthermore, minimizing distractions by choosing a quiet, dedicated study space, and taking regular breaks to avoid b

In [43]:
print(json.dumps(res.tool_calls[0]['args'], indent=2))

{
  "answer": "To study effectively, students should start by setting clear goals and priorities. This involves identifying the most important topics to focus on, breaking down large tasks into smaller manageable chunks, and creating a schedule to stay organized. Active learning techniques such as summarizing notes in their own words, self-quizzing, and elaboration can also help deepen understanding and retention of material. Additionally, using visual aids like diagrams, flowcharts, and mind maps can make complex information more engaging and easier to review. Regular review and practice are crucial, with students aiming to review material within a day or two of initially learning it, and then at increasingly longer intervals to help solidify it in their long-term memory. Furthermore, minimizing distractions by choosing a quiet, dedicated study space, and taking regular breaks to avoid burnout can significantly improve study efficiency. By adopting these strategies, students can devel

In [45]:
revise_instruction = """
Revise your previous answer using the new information.
- You should use the previous critique to add important information to your answer.
You MUST include numerical citations in your revised answer to ensure it can be verified.
- Add a "References" section to the bottom of your answer (which does not count towards the word limit). In form of:
[1] https://example.com
- [2] https://example.com
You should use the previous critique to remove superfluous. information from your answer and make SURE it is not more than 250 words.
"""

In [46]:
revisor_prompt = actor_agent_prompt_template.partial(
    first_instruction = revise_instruction
)

In [47]:
class RevisorClass(AnswerQuestion):
    """Revise original answer to question"""
    reference: List[str] = Field(
        description="Citation motivating updated answer"
    )


In [48]:
revisor_chain = revisor_prompt | llm.bind_tools(
    tools=[RevisorClass], tool_choice="RevisorClass"
)

In [ ]:
## execute tools

In [49]:
from langchain_community.tools import TavilySearchResults

In [50]:
tavily_tool = TavilySearchResults(api_key = "",max_results=5)

/tmp/ipykernel_1331/2436168999.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


ValidationError: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [52]:
from langchain_core.messages import BaseMessage, ToolMessage

In [57]:
def execute_tool(state: List[BaseMessage]) -> List[BaseMessage]:
    last_ai_message: AIMessage = state[-1]

    if not hasattr(last_ai_message, "tool_calls"):
        return []

In [ ]:
## Building the graph

In [53]:
from typing import List
from langgraph.graph import END, MessageGraph

In [54]:
graph = MessageGraph()

/tmp/ipykernel_1331/1254895084.py:1: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = MessageGraph()


In [55]:
graph.add_node("Responder", first_responder_chain)
graph.add_node("Reflexion", revisor_chain)

In [59]:
graph.add_node("execute_tool", execute_tool)

In [60]:
graph.add_edge("Responder","execute_tool")

In [62]:
graph.add_edge("execute_tool", "Reflexion")

In [63]:
MAX_ITERATION = 2

In [64]:
def event_loop(state: List[BaseMessage]) -> str:
    count_tool_visit = sum(
        isinstance(item, ToolMessage) for item in state
    )
    num_iterations = count_tool_visit
    if num_iterations > MAX_ITERATION:
        return END
    return "execute_tool"


In [66]:
graph.add_conditional_edges("Reflexion", event_loop)

In [67]:
graph.set_entry_point("Responder")